# Read interpolated light curves

In [1]:
from snanomaly.dataset.factory import OSCFactory
from snanomaly import dirs
import numpy as np
import polars as pl
from sklearn.manifold import TSNE

from snanomaly.models.results.exception import UnexpectedDataFrameColumnError, EmptyDataFrameError
from snanomaly.models.results.mgp_result import MGPResult


def read_df(path: str) -> pl.DataFrame:
    print(f"Reading from: {path}")
    return pl.scan_parquet(path).collect()

In [2]:
dataset_name = "osc2018_june"
df = read_df(dirs.INTERPOLATED / f"{dataset_name}_LSB-STATIC" / f"{dataset_name}.parquet")
df

Reading from: /home/chongy/stuff/workspace/snanomaly/outputs/interpolated/osc2018_june_LSB-STATIC/osc2018_june.parquet


sn_name,bandset,peak_time,days_pre_peak,days_post_peak,log_likelihood,thetas,pred_means,pred_stds
str,list[str],f64,i64,i64,f64,list[f64],list[list[f64]],list[list[f64]]
"""SDSS-II SN 17907""","[""g_pr"", ""r_pr"", ""i_pr""]",54360.5,20,100,40.859446,"[1.817474, 0.976793, … -0.112782]","[[1.0836e-9, 1.7216e-9, … 0.0], [1.5718e-9, 2.4974e-9, … 0.0], [1.5095e-9, 2.3984e-9, … 0.0]]","[[3.1032e-8, 3.1026e-8, … 0.0], [4.7504e-8, 4.7496e-8, … 0.0], [4.7784e-8, 4.7776e-8, … 0.0]]"
"""SDSS-II SN 15074""","[""g_pr"", ""r_pr"", ""i_pr""]",54019.5,20,100,20.305697,"[1.040616, 0.006335, … -0.185931]","[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0]]","[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0]]"
"""PS1-12bku""","[""g_pr"", ""r_pr"", ""i_pr""]",56206.5,20,100,5.712248,"[1.485435, 1.028905, … -0.212365]","[[3.1984e-9, 4.0253e-9, … 5.2086e-16], [4.3131e-9, 5.2261e-9, … 7.6584e-16], [5.2259e-9, 6.4279e-9, … 9.0266e-16]]","[[3.0114e-9, 2.6499e-9, … 3.5441e-9], [4.6124e-9, 4.0861e-9, … 5.3755e-9], [5.7363e-9, 5.1584e-9, … 6.5915e-9]]"
"""PS1-12bku""","[""g"", ""r"", ""i""]",56174.5,20,100,9.002704,"[0.378619, 0.112686, … -4.6510e-9]","[[1.1954e-14, 2.5198e-13, … 3.3133e-20], [1.8651e-14, 3.9316e-13, … 5.1691e-20], [2.8754e-14, 6.0613e-13, … 7.9693e-20]]","[[2.8726e-9, 2.8726e-9, … 2.8726e-9], [4.5551e-9, 4.5551e-9, … 4.5551e-9], [7.0226e-9, 7.0226e-9, … 7.0226e-9]]"
"""SDSS-II SN 20592""","[""g_pr"", ""r_pr"", ""i_pr""]",54405.5,20,100,26.328436,"[1.245873, 1.124316, … 0.192283]","[[1.3023e-8, 1.1623e-8, … 0.0], [2.2930e-8, 1.9697e-8, … 2.7915e-126], [2.4090e-8, 1.9549e-8, … 4.2657e-126]]","[[4.7809e-13, 1.8088e-10, … 0.0], [4.7809e-13, 1.7267e-9, … 1.4106e-8], [4.7809e-13, 3.8439e-9, … 2.1556e-8]]"
…,…,…,…,…,…,…,…,…
"""SN2016ayf""","[""g"", ""r"", ""i""]",57464.5,20,100,9.115077,"[0.00002, 0.000018, … 0.054203]","[[6.7301e-93, 1.9789e-84, … 0.0], [7.4106e-93, 2.1790e-84, … 0.0], [4.0208e-93, 1.1823e-84, … 0.0]]","[[0.000002, 0.000002, … 0.0], [0.000003, 0.000003, … 0.0], [0.000002, 0.000002, … 0.0]]"
"""SN2007ai""","[""g"", ""r"", ""i""]",54176.5,20,100,61.49233,"[1.616266, 1.287578, … 0.042188]","[[6.4079e-10, 1.1244e-9, … 7.3917e-12], [1.0794e-9, 1.8976e-9, … 1.2469e-11], [8.0077e-10, 1.4090e-9, … 9.2568e-12]]","[[2.1474e-8, 2.1465e-8, … 2.1478e-8], [3.7877e-8, 3.7862e-8, … 3.7884e-8], [2.9635e-8, 2.9624e-8, … 2.9640e-8]]"
"""PTF11dec""","[""g"", ""r"", ""i""]",55704.5,20,100,35.527299,"[2.277329, 1.5342, … 0.059556]","[[2.8156e-9, 3.3589e-9, … 0.0], [4.1483e-9, 4.9497e-9, … 0.0], [2.6185e-9, 3.1252e-9, … 0.0]]","[[7.3147e-9, 7.1975e-9, … 0.0], [1.0879e-8, 1.0708e-8, … 0.0], [7.2381e-9, 7.1353e-9, … 0.0]]"


# t-SNE

In [15]:
def square_distance(X1: np.ndarray, X2: np.ndarray) -> np.ndarray:
    return np.sum((X1 - X2) ** 2)

def rename_numbered_columns(df: pl.DataFrame, new_col_prefix: str) -> pl.DataFrame:
    numeric_cols = [col for col in df.columns if col.isdigit()]
    rename_dict = {col: f"{new_col_prefix}_{col}" for col in numeric_cols}
    return df.rename(rename_dict)

def lists_to_numbered_columns(df: pl.DataFrame, target_col: str) -> pl.DataFrame:
    """Flattens lists of up to 2D lists into columns."""
    # get list dimensions
    first_row = df[target_col][0]
    if isinstance(first_row, pl.Series):
        if isinstance(first_row[0], float):
            num_lists = 1
            list_length = len(first_row)
        elif isinstance(first_row[0], pl.Series):
            num_lists = len(first_row)
            list_length = len(first_row[0])
        else:
            raise UnexpectedDataFrameColumnError(f"Column `{target_col}` must be of type 1D list or 2D list.")
    else:
        raise EmptyDataFrameError(f"No rows in data frame with columns: `{df.columns}`")
    # explode lists until there are no more lists
    nr_dimensions = 1 if num_lists == 1 else 2
    for i in range(nr_dimensions):
        df = df.explode(columns=[target_col])
    df = df.with_row_index(name="col_nr").with_columns(pl.col("col_nr").mod(num_lists * list_length)).pivot(on="col_nr", values=[target_col])
    return rename_numbered_columns(df, target_col)

In [17]:
light_curves = df.select(["sn_name", "bandset", "pred_means", "log_likelihood", "thetas"])
# TODO: band transform if band set differs from `gri`

# transform `pred_means` list of lists into as many columns as there are list elements
light_curves = lists_to_numbered_columns(light_curves, "pred_means")
# transform `thetas` list into columns
light_curves = lists_to_numbered_columns(light_curves, "thetas")
light_curves
# normalize `pred_means` list by flux vector maximum across the bandset


sn_name,bandset,log_likelihood,pred_means_0,pred_means_1,pred_means_2,pred_means_3,pred_means_4,pred_means_5,pred_means_6,pred_means_7,pred_means_8,pred_means_9,pred_means_10,pred_means_11,pred_means_12,pred_means_13,pred_means_14,pred_means_15,pred_means_16,pred_means_17,pred_means_18,pred_means_19,pred_means_20,pred_means_21,pred_means_22,pred_means_23,pred_means_24,pred_means_25,pred_means_26,pred_means_27,pred_means_28,pred_means_29,pred_means_30,pred_means_31,pred_means_32,pred_means_33,…,pred_means_335,pred_means_336,pred_means_337,pred_means_338,pred_means_339,pred_means_340,pred_means_341,pred_means_342,pred_means_343,pred_means_344,pred_means_345,pred_means_346,pred_means_347,pred_means_348,pred_means_349,pred_means_350,pred_means_351,pred_means_352,pred_means_353,pred_means_354,pred_means_355,pred_means_356,pred_means_357,pred_means_358,pred_means_359,pred_means_360,pred_means_361,pred_means_362,thetas_0,thetas_1,thetas_2,thetas_3,thetas_4,thetas_5,thetas_6,thetas_7,thetas_8
str,list[str],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""SDSS-II SN 17907""","[""g_pr"", ""r_pr"", ""i_pr""]",40.859446,1.0836e-9,1.7216e-9,2.6648e-9,4.0185e-9,5.9042e-9,8.4525e-9,1.1792e-8,1.6031e-8,2.1244e-8,2.7442e-8,3.4562e-8,4.2448e-8,5.0851e-8,5.9434e-8,6.7794e-8,7.5495e-8,8.2107e-8,8.7249e-8,9.0623e-8,9.2047e-8,9.1466e-8,8.8946e-8,8.4668e-8,7.8892e-8,7.1939e-8,6.4153e-8,5.5889e-8,4.7494e-8,3.9307e-8,3.1652e-8,2.4841e-8,1.9162e-8,1.4861e-8,1.2104e-8,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.817474,0.976793,0.216847,0.251632,0.365013,0.123047,0.350543,0.120528,-0.112782
"""SDSS-II SN 15074""","[""g_pr"", ""r_pr"", ""i_pr""]",20.305697,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.8829e-10,5.0367e-9,9.8328e-9,1.2632e-8,1.2918e-8,1.1765e-8,1.0756e-8,1.0787e-8,1.1789e-8,1.3273e-8,1.4845e-8,1.6203e-8,1.6939e-8,1.6615e-8,1.5119e-8,1.2903e-8,1.0805e-8,9.5927e-9,9.5749e-9,1.0500e-8,1.1693e-8,1.2308e-8,1.1656e-8,9.5457e-9,6.4879e-9,3.5193e-9,1.6549e-9,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.040616,0.006335,0.4569,0.218178,0.396174,-0.187596,0.436024,-0.233446,-0.185931
"""PS1-12bku""","[""g_pr"", ""r_pr"", ""i_pr""]",5.712248,3.1984e-9,4.0253e-9,4.8097e-9,5.4540e-9,5.8643e-9,5.9704e-9,5.7414e-9,5.1946e-9,4.3937e-9,3.4407e-9,2.4621e-9,1.5934e-9,9.6185e-10,6.6814e-10,7.6591e-10,1.2438e-9,2.0163e-9,2.9318e-9,3.8019e-9,4.4463e-9,4.7424e-9,4.6593e-9,4.2646e-9,3.6983e-9,3.1246e-9,2.6797e-9,2.4341e-9,2.3815e-9,2.4549e-9,2.5586e-9,2.6041e-9,2.5353e-9,2.3380e-9,2.0348e-9,…,2.4533e-9,3.2250e-9,3.9194e-9,4.2073e-9,3.9194e-9,3.2250e-9,2.4533e-9,1.8128e-9,1.3313e-9,9.6421e-10,6.7738e-10,4.5647e-10,2.9360e-10,1.7987e-10,1.0484e-10,5.8104e-11,3.0606e-11,1.5318e-11,7.2846e-12,3.2911e-12,1.4126e-12,5.7605e-13,2.2316e-13,8.2134e-14,2.8719e-14,9.5400e-15,3.0107e-15,9.0266e-16,1.485435,1.028905,0.626331,0.354411,0.521096,-0.131967,0.614195,-0.110177,-0.212365
"""PS1-12bku""","[""g"", ""r"", ""i""]",9.002704,1.1954e-14,2.5198e-13,3.3233e-12,2.7426e-11,1.4172e-10,4.5957e-10,9.4424e-10,1.2778e-9,1.3123e-9,1.4139e-9,1.9055e-9,2.5426e-9,2.6640e-9,1.9441e-9,9.3169e-10,2.9609e-10,1.5105e-10,5.0345e-10,1.6051e-9,3.2424e-9,4.0992e-9,3.2424e-9,1.6046e-9,4.9682e-10,9.6242e-11,1.1664e-11,8.8450e-13,4.1963e-14,1.2456e-15,2.3131e-17,2.6876e-19,1.9537e-21,8.8856e-24,2.5284e-26,…,1.4330e-11,1.0862e-12,5.1526e-14,1.5294e-15,2.8402e-17,3.3000e-19,2.3989e-21,1.0910e-23,3.1046e-26,5.5272e-29,6.1566e-32,4.2904e-35,1.8707e-38,5.1031e-42,8.7095e-46,9.3002e-50,7.7045e-54,2.2322e-50,2.0905e-46,1.2249e-42,4.4906e-39,1.0301e-35,1.4784e-32,1.3278e-29,7.4624e-27,2.6249e-24,5.7797e-22,7.9693e-20,0.378619,0.112686,0.079758

In [ ]:
tsne = TSNE(n_components=)

# UMAP

In [131]:
num_obs = len(df.select(["pred_means"][0])[0])
num_obs

1